In [1]:
import math
import pandas as pd
import joblib
from pathlib import Path
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [3]:
# Dois clientes escalas completamente diferentes
p1 = {"salary": 5000, "historico": 9} # Bom pagador
p2 = {"salary": 5001, "historico": 1} # Pessimo pagador

dx = p1["salary"] - p2["salary"]
dy = p1["historico"] - p2["historico"]
dist = math.sqrt(dx**2 + dy**2)

print(f"Distancia sem normaliação: {dist:.2f}")

Distancia sem normaliação: 8.06


In [16]:
# Knn StandardScaler

df = pd.DataFrame([
    {"salary": 4000, "historico": 8, "aprovado": 1},
    {"salary": 1000, "historico": 2, "aprovado": 0},
    {"salary": 6000, "historico": 9, "aprovado": 1},
    {"salary": 2000, "historico": 3, "aprovado": 0},
    {"salary": 5000, "historico": 7, "aprovado": 1},
    {"salary": 1500, "historico": 1, "aprovado": 0},
    {"salary": 7000, "historico": 2, "aprovado": 1},
    {"salary": 3000, "historico": 1, "aprovado": 0},
    {"salary": 2000, "historico": 8, "aprovado": 1},
    {"salary": 5000, "historico": 5, "aprovado": 0},
    {"salary": 6000, "historico": 9, "aprovado": 1},
    {"salary": 7000, "historico": 4, "aprovado": 0},
    {"salary": 2000, "historico": 3, "aprovado": 1},
    {"salary": 3500, "historico": 2, "aprovado": 0},
    {"salary": 7000, "historico": 3, "aprovado": 1},
    {"salary": 2000, "historico": 5, "aprovado": 0},
])

X = df[["salary", "historico"]]
y = df["aprovado"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Para comparação
X_scaled_df = pd.DataFrame(X_scaled, columns=["salary", "historico"])
X_scaled_df["aprovado"] = y.values

# Treina o KNN
modelo = KNeighborsClassifier(n_neighbors=3)
modelo.fit(X_scaled, y)

print(f"Modelo Treinado com dados Normalizados")
print(X_scaled_df.round(2))

Modelo Treinado com dados Normalizados
    salary  historico  aprovado
0     0.00       1.27         1
1    -1.44      -0.91         0
2     0.96       1.63         1
3    -0.96      -0.54         0
4     0.48       0.91         1
5    -1.20      -1.27         0
6     1.44      -0.91         1
7    -0.48      -1.27         0
8    -0.96       1.27         1
9     0.48       0.18         0
10    0.96       1.63         1
11    1.44      -0.18         0
12   -0.96      -0.54         1
13   -0.24      -0.91         0
14    1.44      -0.54         1
15   -0.96       0.18         0


In [17]:
# Testando dois novos clientes
novos = pd.DataFrame([
    {"salary": 5000, "historico": 9},
    {"salary": 5001, "historico": 1}
])

novo_scaled = scaler.transform(novos)
previsoes = modelo.predict(novo_scaled)

for i, prev in enumerate(previsoes):
    label = "Aprovado" if prev == 1 else "Reprovado"
    print(f"Cliente {i+1} : {label}")

Cliente 1 : Aprovado
Cliente 2 : Reprovado


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_cls = df[["salary", "historico"]]
y_cls = df["aprovado"]

#stratify = mantém a proporção das classes aprovado e reprovado
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=42, stratify=y_cls
)

print("Retorno do train_test_split:")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print()

# Escala com dados de treino e aplica no teste
scaler_cls = StandardScaler()
X_train_s = scaler_cls.fit_transform(X_train)
X_test_s = scaler_cls.transform(X_test)

# Treina e avalia
modelo_cls = KNeighborsClassifier(n_neighbors=3)
modelo_cls.fit(X_train_s, y_train)
y_pred = modelo_cls.predict(X_test_s)

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Reprovado", "Aprovado"],
    zero_division=0,
))

Retorno do train_test_split:
X_train: (12, 2) | X_test: (4, 2)
y_train: (12,) | y_test: (4,)

              precision    recall  f1-score   support

   Reprovado       0.67      1.00      0.80         2
    Aprovado       1.00      0.50      0.67         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4

